# Agentic Workflows: Planning Workflows

A planning-driven research workflow with tool-using research, writing, and editing agents.

## Setup

In [ ]:
from datetime import datetime
import ast
import json
import re
from dotenv import load_dotenv
from IPython.display import Markdown, display
from aisuite import Client
import research_tools

# Load keys from .env and create the shared client.
load_dotenv()
client = Client()

## Exercise 1: Planner agent

In [ ]:
def planner_agent(topic: str, model: str = 'openai:o4-mini') -> list[str]:
    # Create an atomic plan that can be executed by the available agents.
    prompt = f'''
You are a planning agent for a research workflow. Available agents are a tool-using research agent,
a writer agent, and an editor agent. Return ONLY a valid Python list of atomic research-related steps.
Do not include setup, installation, or file-management tasks. The last step must generate a Markdown report.
Topic: {topic}
'''
    response = client.chat.completions.create(
        model=model, messages=[{'role': 'user', 'content': prompt}], temperature=1.0
    )
    steps = ast.literal_eval(response.choices[0].message.content.strip())
    if not isinstance(steps, list) or not all(isinstance(step, str) for step in steps):
        raise ValueError('The planner must return a Python list of strings.')
    return steps

## Exercise 2: Research agent

In [ ]:
def research_agent(task: str, model: str = 'openai:gpt-4o', return_messages: bool = False):
    # Use aisuite's automatic tool loop for evidence collection.
    print('==================================')
    print('🔍 Research Agent')
    print('==================================')
    prompt = f'''
You are a research assistant with arxiv_search_tool for papers, tavily_search_tool for web sources,
and wikipedia_search_tool for background knowledge. Use tools when useful, cite URLs, and never invent sources.
Task: {task}
Today is {datetime.now().strftime('%Y-%m-%d')}.
'''
    messages = [{'role': 'user', 'content': prompt.strip()}]
    tools = [research_tools.arxiv_search_tool, research_tools.tavily_search_tool, research_tools.wikipedia_search_tool]
    try:
        response = client.chat.completions.create(
            model=model, messages=messages, tools=tools, tool_choice='auto', max_turns=12
        )
        content = response.choices[0].message.content
        print('✅ Output:\n', content)
        return (content, messages) if return_messages else content
    except Exception as error:
        print('❌ Error:', error)
        return f'[Model Error: {error}]'

## Exercise 3: Writer agent

In [ ]:
def writer_agent(task: str, model: str = 'openai:gpt-4o') -> str:
    # Draft or summarize academic and technical content.
    print('==================================')
    print('✍️ Writer Agent')
    print('==================================')
    messages = [
        {'role': 'system', 'content': 'You are a writing agent specialized in generating well-structured academic or technical content.'},
        {'role': 'user', 'content': task},
    ]
    response = client.chat.completions.create(model=model, messages=messages, temperature=1.0)
    return response.choices[0].message.content

## Exercise 4: Editor agent

In [ ]:
def editor_agent(task: str, model: str = 'openai:gpt-4o') -> str:
    # Reflect on, critique, or improve an existing draft.
    print('==================================')
    print('🧠 Editor Agent')
    print('==================================')
    messages = [
        {'role': 'system', 'content': 'You are an editor agent. Your job is to reflect on, critique, or improve existing drafts.'},
        {'role': 'user', 'content': task},
    ]
    response = client.chat.completions.create(model=model, messages=messages, temperature=0.7)
    return response.choices[0].message.content

## Exercise 5: Executor agent

In [ ]:
agent_registry = {'research_agent': research_agent, 'writer_agent': writer_agent, 'editor_agent': editor_agent}


def clean_json_block(raw: str) -> str:
    # Remove optional Markdown fences before parsing the JSON response.
    raw = raw.strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```(?:json)?\n?', '', raw)
        raw = re.sub(r'\n?```$', '', raw)
    return raw.strip()


def executor_agent(plan_steps: list[str], model: str = 'openai:gpt-4o'):
    # Route every plan step and pass outputs from earlier steps as context.
    history = []
    for step in plan_steps:
        prompt = f'''
You are an execution manager for a multi-agent research team.
Return only JSON with agent (research_agent, writer_agent, or editor_agent) and task keys.
Instruction: {step}
'''
        decision = client.chat.completions.create(
            model=model, messages=[{'role': 'user', 'content': prompt}], temperature=0
        )
        agent_info = json.loads(clean_json_block(decision.choices[0].message.content))
        agent_name, task = agent_info['agent'], agent_info['task']
        context = '\n'.join(f'Step {i + 1} executed by {agent}:\n{result}' for i, (_, agent, result) in enumerate(history))
        enriched_task = f'You are {agent_name}.\n\nPrevious context:\n{context}\n\nYour next task:\n{task}'
        print(f'\n🛠️ Executing with agent: `{agent_name}` on task: {task}')
        if agent_name not in agent_registry:
            raise ValueError(f'Unknown agent: {agent_name}')
        output = agent_registry[agent_name](enriched_task)
        history.append((step, agent_name, output))
        print(f'✅ Output:\n{output}')
    return history

## Full research-report workflow

In [ ]:
def generate_research_report(prompt: str) -> dict:
    # Orchestrate planning, dynamic research, drafting, feedback, and revision.
    plan = planner_agent(prompt)
    research_task = 'Gather evidence for this topic using this plan:\n' + '\n'.join(f'- {step}' for step in plan)
    research = research_agent(research_task)
    draft = writer_agent(f'Write a short sourced Markdown report on {prompt}. Use this evidence:\n{research}')
    feedback = editor_agent(f'Give concise strengths, limitations, and revision suggestions for this draft:\n{draft}')
    revised_report = writer_agent(f'Revise this report using the editor feedback. Preserve supported claims and source URLs.\n\nDraft:\n{draft}\n\nFeedback:\n{feedback}')
    return {'plan': plan, 'research': research, 'draft': draft, 'feedback': feedback, 'revised_report': revised_report}


# Run after adding both keys to .env.
report = generate_research_report('The ensemble Kalman filter for time series forecasting')
display(Markdown(report['revised_report']))